# <div align='center'>🌍 Cidades Inteligentes: Qualidade do Ar em Bogotá (RMCAB) </div>

![Banner](https://raw.githubusercontent.com/JamaicaStoAndre/Ci-ncia-de-dados-com-Pandas/master/bogota_air_quality_banner.png)

--- 

## 🎓 Contexto: IA na Borda - Mestrado UFSC

Este material faz parte do projeto de monitoramento da **Rede de Monitoramento da Qualidade do Ar de Bogotá (RMCAB)**. Aqui, exploramos a **Biblioteca Pandas** como a ferramenta central da **Fase B (Design e Preparação)** da nossa arquitetura padronizada de Ciência de Dados.

### 🏛️ As Três Fases do Projeto

1. **Fase A - Explorar (Discovery)**: 
   * **Objetivo**: Avaliar a viabilidade técnica e entender o domínio dos dados de sensores.
   * **Ação**: Identificar se os sensores estão ativos e se os dados são acessíveis.

2. **Fase B - Design (Preparação)**: 
   * **Objetivo**: Limpar, transformar e estruturar os dados brutos para alimentar modelos de IA.
   * **Ação**: Tratamento de nulos, filtros, agrupamentos e engenharia de variáveis (Pandas).

3. **Fase C - Implementar (Prototipagem)**: 
   * **Objetivo**: Criar a solução final e modelos de Machine Learning (como KNN ou Redes Neurais).
   * **Ação**: Geração de visualizações geoespaciais e predições de poluição.

---

In [ ]:
import pandas as pd           # Importa pandas para manipulação de tabelas (DataFrames)
import numpy as np            # Importa numpy para operações numéricas e tratamento de NaN
import matplotlib.pyplot as plt # Importa matplotlib para criação de gráficos estáticos
import seaborn as sns          # Importa seaborn para visualizações estatísticas avançadas
import folium                 # Importa folium para criar mapas interativos no navegador
from folium.plugins import HeatMap # Importa a função de mapa de calor do folium
import re                     # Importa re para busca e limpeza de padrões em texto
import os                     # Importa os para verificar caminhos e existência de arquivos

def parse_dms(coor):
    """Converte coordenadas DMS (Strings) para formato Decimal (Floats)"""
    if not isinstance(coor, str): return coor # Se não for texto, retorna o valor original
    parts = re.split(r'[^\d\w]+', coor)   # Divide o texto ignorando caracteres especiais
    degrees = float(parts[0])               # Extrai os graus como número flutuante
    minutes = float(parts[1])               # Extrai os minutos como número flutuante
    seconds = float(parts[2]+'.'+parts[3])  # Concatena segundos e frações e converte
    direction = parts[4]                    # Identifica a Rosa dos Ventos (N, S, E, W)
    dec_coord = degrees + minutes / 60 + seconds / 3600 # Faz o cálculo matemático decimal
    if direction in ['S', 'W']:             # Se a direção for Sul ou Oeste...
        dec_coord *= -1                     # ...torna o valor da coordenada negativo
    return dec_coord                        # Retorna a coordenada decimal pronta para o mapa

print("✅ Módulos carregados e utilitários configurados!") # Log de sucesso

## 📂 1. Ingestão de Dados (Fase B - Início)

O objetivo aqui é carregar os dados brutos dos sensores para a memória ram do computador.

In [ ]:
USE_DRIVE = False                             # Variável de controle para uso do Google Drive
if USE_DRIVE:                                 # Se optarmos por usar o Drive...
    from google.colab import drive           # Importa a biblioteca de montagem do Drive
    drive.mount('/content/drive')            # Monta o armazenamento em nuvem no Colab
    csv_path = '/content/drive/MyDrive/1-MestradoUFSC/IA_NA_BORDA/bogota_sensors_sample.csv' # Caminho no Drive
else:                                         # Caso contrário (Upload local)...
    opcoes = ['bogota_sensors_sample.csv', 'Bogota_sensors_sample.csv', '/home/Bogota_sensors_sample.csv'] # Lista de busca
    csv_path = next((p for p in opcoes if os.path.exists(p)), 'bogota_sensors_sample.csv') # Seleciona o primeiro válido

if os.path.exists(csv_path):                  # Verifica se o arquivo final realmente existe...
    df = pd.read_csv(csv_path)                # Lê o CSV e o transforma em um DataFrame do Pandas
    df['Lat_Dec'] = df['Latitude'].apply(parse_dms) # Aplica conversão de coordenadas Latitude
    df['Lon_Dec'] = df['Longitude'].apply(parse_dms) # Aplica conversão de coordenadas Longitude
    print(f"🚀 Sucesso! Colunas lidas: {df.columns.tolist()}") # Log das colunas encontradas
    display(df.head())                        # Exibe as primeiras 5 linhas da nossa tabela
else:                                         # Caso o arquivo não seja encontrado...
    print("❗ ERRO: Arquivo não encontrado na busca automática.") # Alerta o usuário

## 📊 2. Análise e Estatística (Fase B - Exploração Técnica)

Calculamos métricas para entender a severidade da poluição em cada localidade.

In [ ]:
if 'df' in locals():                          # Proteção: só executa se o DataFrame 'df' existir
    resumo = df[['PM2.5', 'PM10', 'CO']].describe() # Calcula média, min, max, std dos poluentes
    print("--- Estatísticas Gerais ---")      # Título informativo
    display(resumo)                           # Exibe a tabela de resumo estatístico
    
    agrupado = df.groupby('Station')[['PM2.5', 'PM10']].mean() # Agrupa por Estação e calcula a média
    agrupado = agrupado.sort_values(by='PM2.5', ascending=False) # Ordena de forma decrescente
    print("\n--- Médias de Poluição por Localidade ---") # Outro título informativo
    display(agrupado)                          # Exibe o ranking de poluição por bairro/estação

## ⚠️ 3. Filtros de Alerta (Fase B - Tomada de Decisão)

Identificamos anomalias de poluição que exigem ação imediata.

In [ ]:
if 'df' in locals():                          # Verifica existência dos dados
    limiar = 40                                # Define um valor de corte para PM2.5 (Alerta)
    mascara = df['PM2.5'] > limiar             # Cria um filtro booleano (Onde PM2.5 > limiar?)
    alertas = df[mascara]                      # Filtra a tabela original mantendo apenas os alertas
    print(f"🔥 Temos {len(alertas)} horas com ar em estado crítico.") # Exibe contagem de registros
    display(alertas[['DateTime', 'Station', 'PM2.5']].head()) # Mostra detalhes dos primeiros alertas

## 🤖 4. Preparação para Machine Learning (Fase B - Final)

Pré-processamento: Transformamos dados categóricos e temporais em números (Input da IA).

In [ ]:
if 'df' in locals():                          # Verifica se os dados estão carregados
    df['DateTime'] = pd.to_datetime(df['DateTime']) # Converte coluna de texto para formato Data/Hora
    df['hora'] = df['DateTime'].dt.hour       # Extrai apenas a hora do dia (0-23)
    df['dia_semana'] = df['DateTime'].dt.dayofweek # Extrai o dia da semana (0=Segunda)
    
    # Transforma texto das Estações em colunas binárias zeros e uns (Dummies)
    df_ml = pd.get_dummies(df, columns=['Station'], prefix='S')
    
    print("🔬 Dataset pronto para treino da Fase C!") # Log de conclusão de pré-processamento
    display(df_ml.head())                     # Exibe a tabela final espalhada (Sparse Matrix)

## 🗺️ 5. Visualização (Fase C - Prototipagem)

O objetivo é gerar clareza visual sobre as zonas de risco geoespacial.

In [ ]:
if 'df' in locals():                          # Verifica se os dados existem
    m = folium.Map(location=[4.65, -74.1], zoom_start=12) # Cria o mapa base centrado em Bogotá
    folium.TileLayer('cartodbpositron').add_to(m) # Adiciona um estilo moderno e limpo ao mapa
    calc_calor = [[r['Lat_Dec'], r['Lon_Dec'], r['PM2.5']] for i, r in df.iterrows()] # Cria lista de pontos térmicos
    HeatMap(calc_calor, radius=18).add_to(m)   # Adiciona a mancha de calor ao mapa
    print("🌍 Mapa Gerado com Sucesso!")        # Log de finalização do mapa
    display(m)                                 # Renderiza o mapa interativo na tela

---

## 🎓 Desafio Final (Estudante)

1. **Mínimos**: Qual a estação com o menor valor de PM10? 
2. **Filtro Local**: Mostre apenas o histórico da estação 'USM' com PM2.5 > 25.

### ✅ Resolução do Desafio (Spoilers)

In [ ]:
# Resolução 1: Encontrando a estação mais limpa
estacao_limpa = df.groupby('Station')['PM10'].min().idxmin() # Agrupa, pega o min e o nome da estacao
print(f"⭐ A estação com o ar mais limpo (mínimo PM10) é: {estacao_limpa}")

# Resolução 2: Filtro específico para USM
usm_filtrado = df[(df['Station'] == 'USM') & (df['PM2.5'] > 25)] # Aplica dois filtros simultâneos
print(f"🔍 Registros críticos em USM: {len(usm_filtrado)}")
display(usm_filtrado.head())